# Arxiv (2026년 최신 권장 사용법)

[arXiv](https://arxiv.org/)은 물리학, 수학, 컴퓨터 과학, 정량 생물학, 정량 금융, 통계, 전기공학 및 시스템 과학, 경제학 분야의 학술 논문을 위한 오픈 액세스 아카이브입니다.

> **⚠️ 2026년 9월 기준 변경 사항 (공통)**
>
> - 책에서 사용한 `langchain_community.document_loaders` 는 **`langchain-community` 패키지 sunset**(2026년 5월 발표, 6월 저장소 아카이브)으로 더 이상 유지보수되지 않습니다. 설치·import 는 여전히 되지만 새 프로젝트의 기반으로 권장되지 않습니다.
> - LangChain 의 현재 방향은 ① **전용 통합 패키지**(`langchain-upstage`, `langchain-unstructured`, `langchain-pymupdf4llm`, `langchain-docling` 등)를 쓰거나, ② 파싱 라이브러리를 **직접 사용**하고 결과를 `langchain_core.documents.Document` 로 감싸는 것(필요하면 `BaseLoader` 를 상속한 작은 로더 클래스 작성)입니다.
> - `Document`, `BaseLoader`, 텍스트 분할기(`langchain-text-splitters`)는 그대로 유지되는 핵심 인터페이스입니다.

**이 노트북에서 바뀐 점**

| 책(구버전, `langchain_community`) | 현재 권장 |
|---|---|
| `ArxivLoader(query, load_max_docs, load_all_available_meta)` | **`arxiv`** 패키지로 검색 + **`pymupdf`** 로 PDF 텍스트 추출 → `Document` 로더 직접 작성 |
| `loader.get_summaries_as_docs()` | 초록(summary)만으로 `Document` 생성 (PDF 다운로드 없음 → 빠름) |

`ArxivLoader` 가 내부에서 하던 일(검색 → PDF 다운로드 → PyMuPDF 로 텍스트 변환 → 메타데이터 구성)을 그대로 구현합니다.

In [ ]:
# 설치
# !pip install -qU langchain-core arxiv pymupdf

## 로더 구현

- 메타데이터 키는 책의 `ArxivLoader` 와 동일하게 `Published`, `Title`, `Authors`, `Summary` 를 기본으로 사용합니다.
- `load_all_available_meta=True` 이면 `entry_id`, `categories`, `links` 등 추가 메타데이터를 넣습니다.
- arXiv API 이용 정책상 요청 간 지연(`delay_seconds`)을 두는 것이 좋습니다.

In [ ]:
import urllib.request
from typing import Iterator

import arxiv
import pymupdf
from langchain_core.document_loaders import BaseLoader
from langchain_core.documents import Document


class ArxivPaperLoader(BaseLoader):
    """arxiv 검색 결과를 Document 로 변환 (본문 전체 또는 초록만)"""

    def __init__(
        self,
        query: str,
        load_max_docs: int = 3,
        load_all_available_meta: bool = False,
        summaries_only: bool = False,
    ) -> None:
        self.query = query
        self.load_max_docs = load_max_docs
        self.load_all_available_meta = load_all_available_meta
        self.summaries_only = summaries_only
        self.client = arxiv.Client(page_size=load_max_docs, delay_seconds=3.0, num_retries=3)

    def _metadata(self, r: arxiv.Result) -> dict:
        meta = {
            "Published": str(r.published.date()),
            "Title": r.title,
            "Authors": ", ".join(a.name for a in r.authors),
            "Summary": r.summary,
        }
        if self.load_all_available_meta:
            meta.update(
                {
                    "entry_id": r.entry_id,
                    "published_first_time": str(r.published.date()),
                    "updated": str(r.updated.date()),
                    "comment": r.comment,
                    "journal_ref": r.journal_ref,
                    "doi": r.doi,
                    "primary_category": r.primary_category,
                    "categories": r.categories,
                    "links": [link.href for link in r.links],
                }
            )
        return meta

    @staticmethod
    def _pdf_text(pdf_url: str) -> str:
        request = urllib.request.Request(pdf_url, headers={"User-Agent": "arxiv-loader-example"})
        with urllib.request.urlopen(request, timeout=60) as resp:
            pdf_bytes = resp.read()
        with pymupdf.open(stream=pdf_bytes, filetype="pdf") as pdf:
            return "".join(page.get_text() for page in pdf)

    def lazy_load(self) -> Iterator[Document]:
        search = arxiv.Search(
            query=self.query,
            max_results=self.load_max_docs,
            sort_by=arxiv.SortCriterion.Relevance,
        )
        for result in self.client.results(search):
            metadata = self._metadata(result)
            if self.summaries_only:
                yield Document(page_content=result.summary, metadata=metadata)
            else:
                yield Document(page_content=self._pdf_text(result.pdf_url), metadata=metadata)

    def get_summaries_as_docs(self) -> list[Document]:
        """책의 ArxivLoader.get_summaries_as_docs() 대응"""
        previous, self.summaries_only = self.summaries_only, True
        try:
            return self.load()
        finally:
            self.summaries_only = previous

## 객체 생성

이제 로더 객체를 인스턴스화하고 문서를 로드할 수 있습니다.

In [ ]:
# query 에 검색하고자 하는 논문의 주제를 입력합니다.
loader = ArxivPaperLoader(
    query="Chain of thought",
    load_max_docs=2,  # 최대 문서 수
    load_all_available_meta=True,  # 메타데이터 전체 로드 여부
)

In [ ]:
# 문서 로드 결과출력
docs = loader.load()
docs

In [ ]:
# 문서 메타데이터 출력
docs[0].metadata

`load_all_available_meta=False` 인 경우 메타데이터는 전체가 아닌 일부(`Published`, `Title`, `Authors`, `Summary`)만 출력됩니다.

In [ ]:
# query 에 검색하고자 하는 논문의 주제를 입력합니다.
loader = ArxivPaperLoader(
    query="ChatGPT",
    load_max_docs=2,  # 최대 문서 수
    load_all_available_meta=False,  # 메타데이터 전체 로드 여부
)

# 문서 로드 결과출력
docs = loader.load()

# 문서 메타데이터 출력
docs[0].metadata

## 요약(summary)

- 논문의 전체 내용이 아닌 요약본(초록)만 필요하다면 `get_summaries_as_docs()` 를 호출합니다. PDF 를 내려받지 않으므로 훨씬 빠릅니다.

In [ ]:
# 문서 요약 로딩
docs = loader.get_summaries_as_docs()

# 첫 번째 문서 접근
print(docs[0].page_content)

## lazy_load()

문서를 대량으로 로드할 때 일부 문서에 대해 먼저 후속 작업을 할 수 있다면, 메모리 사용량을 줄이기 위해 문서를 한 번에 하나씩 지연 로드할 수 있습니다.

In [ ]:
docs = []

# 문서 지연 로드
for doc in loader.lazy_load():
    docs.append(doc)

In [ ]:
# 결과 출력
docs

> 💡 **arXiv 논문을 더 정확하게 파싱하고 싶다면** PDF URL 을 `PyMuPDF4LLMLoader`(Markdown 출력), `DoclingLoader`, `UpstageDocumentParseLoader` 같은 레이아웃 인식 파서에 넘기는 방법도 있습니다. 수식·표가 많은 논문에서 품질 차이가 큽니다.